# ARC Transformer — Colab

**Before running:** Runtime → Change runtime type → GPU (T4 or better).

Each cell defines `REPO` itself, so cells can be run independently after a restart.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — change runtime type to GPU"
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## (Optional) Mount Google Drive
Uncomment to persist checkpoints across sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('/content/drive/MyDrive/arc-agi-checkpoints', exist_ok=True)

## Setup — clone repo and download data

In [ ]:
import os

REPO = '/content/arc-agi-solver'

if not os.path.exists(REPO):
    !git clone https://github.com/rodehyde/arc-agi-solver.git {REPO}
else:
    !git -C {REPO} pull

os.chdir(REPO)
print(f"Working directory: {os.getcwd()}")

In [ ]:
REPO = '/content/arc-agi-solver'

# RE-ARC synthetic data (~400 MB) — needed for training
if not os.path.exists(f'{REPO}/data/re_arc'):
    !python {REPO}/scripts/download_re_arc.py
else:
    print('RE-ARC already present')

In [ ]:
REPO = '/content/arc-agi-solver'

# Original ARC tasks (~1 MB) — needed for evaluation
if not os.path.exists(f'{REPO}/data/training'):
    !git clone --depth 1 https://github.com/fchollet/ARC-AGI.git /tmp/arc-agi-repo
    !mkdir -p {REPO}/data
    !cp -r /tmp/arc-agi-repo/data/training {REPO}/data/training
    !cp -r /tmp/arc-agi-repo/data/evaluation {REPO}/data/evaluation
    print('ARC data downloaded')
else:
    print('ARC data already present')

## Train — initial 50-epoch run

In [ ]:
REPO = '/content/arc-agi-solver'

!mkdir -p {REPO}/logs {REPO}/checkpoints
!python {REPO}/scripts/train_transformer.py \
    --clusters 18 \
    --epochs 50 \
    --steps-per-epoch 200 \
    --lr 1e-4 \
    --save-every 10 \
    --log-every 5 \
    --patience 4 \
    --warmup-epochs 3 \
    --log {REPO}/logs/train_transformer_c18.log

## Train — extend from best checkpoint

Run this after the initial 50-epoch run to continue from the best checkpoint.

In [ ]:
import os

REPO = '/content/arc-agi-solver'
best_path = f'{REPO}/checkpoints/transformer_c18_best.pt'

assert os.path.exists(best_path), f"Best checkpoint not found at {best_path}"
print(f"Resuming from: {best_path}")

!python {REPO}/scripts/train_transformer.py \
    --clusters 18 \
    --epochs 150 \
    --steps-per-epoch 200 \
    --lr 1e-4 \
    --save-every 10 \
    --log-every 5 \
    --patience 6 \
    --warmup-epochs 3 \
    --log {REPO}/logs/train_transformer_c18_ext.log \
    --resume {best_path}

## Evaluate — greedy decoding on original ARC tasks

Loads the best checkpoint and runs greedy generation on the 20 cluster-18
tasks using their actual training pairs as context. Displays input, predicted
output, and expected output side by side.

In [ ]:
import json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

REPO = '/content/arc-agi-solver'
sys.path.insert(0, REPO)

from src.arc_tokenizer import ArcTokenizer, VOCAB_SIZE
from src.transformer_model import ArcTransformer

ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

# ------------------------------------------------------------------
# Load model
# ------------------------------------------------------------------
best_path = f'{REPO}/checkpoints/transformer_c18_best.pt'
assert os.path.exists(best_path), f"No checkpoint at {best_path}"

ckpt = torch.load(best_path, map_location='cpu')
args = ckpt['args']
task_ids = ckpt['task_ids']
print(f"Checkpoint: epoch {ckpt['epoch']+1}, best_val_loss={ckpt.get('best_val_loss', float('nan')):.4f}")
print(f"Tasks ({len(task_ids)}): {task_ids}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ArcTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=args['d_model'],
    n_heads=args['n_heads'],
    n_layers=args['n_layers'],
    max_seq_len=args['max_seq_len'],
    max_grid_dim=args['max_grid_dim'],
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()
print("Model loaded")

tokenizer = ArcTokenizer()
K = args.get('k_context', 3)

# ------------------------------------------------------------------
# Greedy evaluation
# ------------------------------------------------------------------
def evaluate_task(task_id):
    task = json.load(open(f'{REPO}/data/training/{task_id}.json'))
    train_pairs = [(np.array(p['input'],  dtype=np.uint8),
                    np.array(p['output'], dtype=np.uint8))
                   for p in task['train']]
    test_inp = np.array(task['test'][0]['input'],  dtype=np.uint8)
    test_out = np.array(task['test'][0]['output'], dtype=np.uint8)

    ctx = train_pairs[:K]
    feats, _ = tokenizer.encode_sequence(ctx, test_inp, test_output=None)
    feats_t  = torch.from_numpy(feats).unsqueeze(0).long().to(device)
    pad_mask = torch.zeros(1, feats_t.shape[1], dtype=torch.bool, device=device)

    grid_num = 2 * len(ctx) + 2
    pred = model.generate(feats_t, pad_mask,
                          test_out.shape[0], test_out.shape[1], grid_num)

    exact = bool(np.array_equal(pred, test_out))

    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    for ax, grid, title in zip(axes,
                                [test_inp, pred, test_out],
                                ['Input', 'Predicted', 'Expected']):
        ax.imshow(grid, cmap=CMAP, norm=NORM, interpolation='nearest')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    match_str = '✓ exact match' if exact else '✗'
    fig.suptitle(f"{task_id}  —  {match_str}", fontsize=11)
    plt.tight_layout()
    plt.show()
    return exact

n_correct = 0
for tid in task_ids:
    n_correct += evaluate_task(tid)

print(f"\nExact match: {n_correct} / {len(task_ids)} tasks")

## Show training log

In [ ]:
REPO = '/content/arc-agi-solver'

for log in [f'{REPO}/logs/train_transformer_c18.log',
            f'{REPO}/logs/train_transformer_c18_ext.log']:
    if os.path.exists(log):
        print(f'=== {log} ===')
        !cat {log}
        print()

## Download checkpoints

In [ ]:
import glob, os
from google.colab import files

REPO = '/content/arc-agi-solver'

for p in sorted(glob.glob(f'{REPO}/checkpoints/transformer_c18*.pt')):
    print(p)

# Uncomment to download:
# files.download(f'{REPO}/checkpoints/transformer_c18_best.pt')